# Build D-bias / D-align from GenImage SD v1.4

Implements Section III-C (*Dataset construction*) of the paper. Both training versions come from **one shared identifier list** drawn from the Stable Diffusion v1.4 partition of GenImage:

- **train**: 10 real + 10 synthetic images per ImageNet class (10,000 each)
- **val**: 2 real + 2 synthetic per class (2,000 each), used only for learning-rate and threshold selection and for the encoding checks

| | Real images (identical files in both versions) | Synthetic images |
|---|---|---|
| **D-bias** | JPEG, lossless DCT-domain crop to 336×336 (original quantization tables + chroma subsampling, one compression pass), metadata stripped | PNG, 336×336 crop |
| **D-align** | same files | the same crop, JPEG-encoded **once** with the quantization tables + subsampling of a randomly paired real image of the same class |

Synthetic images are cropped at an offset whose remainder modulo 8 is neither 0 nor 4 on either axis, so the 8-pixel pattern of the Stable Diffusion autoencoder lines up with neither the JPEG block grid nor the 4-pixel-shifted grid used in encoding check 2.

Also applied: real images must have a shorter side ≥ 336, and near-duplicates are removed with a perceptual hash. Two encoding checks follow, each trained on train and scored on val:
1. **Metadata:** a logistic regression on format, estimated quality factor, quantization-table hash, subsampling and dimensions must reach ≥ 0.95 accuracy on D-bias and ≤ 0.55 on D-align.
2. **Pixels (grid shift):** JPEG features (blockiness + DCT coefficient statistics) are computed on the 8×8 grid and on the grid shifted by 4 px, with one logistic regression on each. D-align passes if accuracy(grid) − accuracy(shifted) ≤ 0.05.

**Kaggle setup:** add the dataset `vtphatt2/genimage-stable-diffusion-v1-4` as input and turn **Internet on** (needed for `apt-get install libjpeg-turbo-progs`, plus the ImageNet val labels if the val reals have no class in their filenames).

**Run once with `SMOKE_CLASSES = 5`** (takes about a minute) and check the output, then set it to `None` for the full build.

Outputs in `/kaggle/working/genimage_sd14_336/`:

| Path | Content |
|---|---|
| `real/{split}/{class}/*.jpg` | real images, shared by both versions |
| `fake_png/{split}/{class}/*.png` | synthetic images for D-bias |
| `fake_jpeg/{split}/{class}/*.jpg` | synthetic images for D-align |
| `d_bias.csv`, `d_align.csv` | `id, path, label, split, class_idx`: what the training loaders read |
| `manifest.csv` | one row per image with provenance, crop box, pHash, pairing and hashes |
| `selection_rejects.csv`, `metadata_features.csv`, `grid_features.npz`, `audit_metadata.json`, `audit_grid_shift.json`, `build_info.json` | logs and audits |

## 1. Configuration

In [ ]:
import io, os, re, json, time, zlib, hashlib, shutil, subprocess, platform, urllib.request
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import PIL
from PIL import Image, JpegImagePlugin, features
from PIL.JpegImagePlugin import get_sampling
from scipy.fft import dct
from tqdm.auto import tqdm

INPUT_ROOT = Path("/kaggle/input")
OUT = Path("/kaggle/working/genimage_sd14_336")

# Optional overrides if auto-discovery fails:
# {("train", "ai"): Path(...), ("train", "nature"): ..., ("val", "ai"): ..., ("val", "nature"): ...}
SPLIT_DIRS = None
# ImageNet val ground truth (one wnid per line, 50,000 lines). Used only if the val reals are
# named ILSVRC2012_val_XXXXXXXX.JPEG, with no class in the name.
VAL_LABELS_PATH = None
VAL_LABELS_URL = ("https://raw.githubusercontent.com/tensorflow/models/master/"
                  "research/slim/datasets/imagenet_2012_validation_synset_labels.txt")

SEED = 0
N_PER_CLASS = {"train": 10, "val": 2}  # per class and per label
CROP = 336
FAKE_OFFSET_RESIDUES = (1, 2, 3, 5, 6, 7)  # allowed synthetic crop offsets mod 8 (never 0 or 4)
GRID_SHIFT = 4         # shift of the comparison grid in encoding check 2
PHASH_MAX_DIST = 6    # 64-bit pHash Hamming distance at or below which two images are near-duplicates
SMOKE_CLASSES = 5      # set to None for the full 1,000-class build
N_WORKERS = min(16, 2 * (os.cpu_count() or 2))
CHECK_PIXELS = True    # check that every real crop decodes to the same pixels as the source
# GenImage's val split has only a few reals per class. When a val class runs short after the checks,
# fill it from unused images of the same class and label in the GenImage train folders (logged as src_split).
FILL_VAL_FROM_TRAIN = True

T0 = time.time()
OUT.mkdir(parents=True, exist_ok=True)

## 2. Tools and versions
`jpegtran` (libjpeg-turbo) does the lossless DCT-domain crop of the real images.

In [ ]:
def ensure_jpegtran():
    path = shutil.which("jpegtran")
    if path is None:
        subprocess.run("apt-get -qq update && apt-get -qq install -y libjpeg-turbo-progs",
                       shell=True, check=True)
        path = shutil.which("jpegtran")
    if path is None:
        raise RuntimeError("jpegtran not found: install libjpeg-turbo-progs (on Kaggle, turn Internet on)")
    return path

JPEGTRAN = ensure_jpegtran()
_v = subprocess.run([JPEGTRAN, "-version"], capture_output=True, text=True,
                    stdin=subprocess.DEVNULL, timeout=10)
_v = (_v.stderr or _v.stdout).strip().splitlines()

BUILD_INFO = {
    "python": platform.python_version(),
    "pillow": PIL.__version__,
    "pillow_libjpeg": features.version("jpg"),
    "numpy": np.__version__,
    "jpegtran": _v[0] if _v else "unknown",
    "seed": SEED, "n_per_class": N_PER_CLASS, "crop": CROP,
    "phash_max_dist": PHASH_MAX_DIST, "fake_offset_residues": FAKE_OFFSET_RESIDUES,
    "grid_shift": GRID_SHIFT, "smoke_classes": SMOKE_CLASSES,
}
print(json.dumps(BUILD_INFO, indent=2))

## 3. Find the data and label every image with its ImageNet class
GenImage layout: `.../imagenet_ai_0419_sdv4/{train,val}/{ai,nature}/`.
- Synthetic: `583_sdv4_00020.png`, so the class index is the prefix.
- Real: ImageNet files `n01440764_10026.JPEG`; the wnid maps to the class index through the sorted 1,000 wnids (standard ImageNet order). Val reals named `ILSVRC2012_val_*.JPEG` are labelled from the ImageNet val ground truth.

In [ ]:
KEYS = [("train", "ai"), ("train", "nature"), ("val", "ai"), ("val", "nature")]
IMG_EXT = {".jpg", ".jpeg", ".png"}
WNID_RE = re.compile(r"(n\d{8})")
VAL_RE = re.compile(r"ILSVRC2012_val_(\d{8})")
AI_RE = re.compile(r"^(\d+)_")


def find_split_dirs(root):
    hits = {}
    for dirpath, dirnames, _ in os.walk(root):
        d = Path(dirpath)
        if d.name in ("train", "val"):
            for kind in ("ai", "nature"):
                if kind in dirnames:
                    hits.setdefault((d.name, kind), []).append(d / kind)
            dirnames[:] = [n for n in dirnames if n not in ("ai", "nature")]
    return hits


def list_images(d):
    out = []
    for dirpath, _, files in os.walk(d):
        out += [os.path.join(dirpath, f) for f in files if os.path.splitext(f)[1].lower() in IMG_EXT]
    return sorted(out)


if SPLIT_DIRS is None:
    hits = find_split_dirs(INPUT_ROOT)
    for k in KEYS:
        print(k, [str(p) for p in hits.get(k, [])])
    missing = [k for k in KEYS if k not in hits]
    ambiguous = [k for k in KEYS if len(hits.get(k, [])) > 1]
    if missing or ambiguous:
        raise RuntimeError(f"Set SPLIT_DIRS by hand. missing={missing} ambiguous={ambiguous}")
    SPLIT_DIRS = {k: hits[k][0] for k in KEYS}

files = {k: list_images(SPLIT_DIRS[k]) for k in KEYS}
for k in KEYS:
    print(f"{k}: {len(files[k]):,} files, e.g. {[os.path.basename(p) for p in files[k][:3]]}")

root_nature = SPLIT_DIRS[("train", "nature")]
WNIDS = sorted({m.group(1) for p in files[("train", "nature")]
                if (m := WNID_RE.search(os.path.relpath(p, root_nature)))})
if len(WNIDS) != 1000:
    print(f"WARNING: expected 1000 ImageNet wnids in train/nature, found {len(WNIDS)}; "
          "class indices of reals may not match the synthetic prefixes")
WNID_TO_IDX = {w: i for i, w in enumerate(WNIDS)}

_val_labels = None
def val_labels():
    global _val_labels
    if _val_labels is None:
        if VAL_LABELS_PATH:
            text = Path(VAL_LABELS_PATH).read_text()
        else:
            text = urllib.request.urlopen(VAL_LABELS_URL, timeout=60).read().decode()
        _val_labels = text.split()
        assert len(_val_labels) == 50000, len(_val_labels)
    return _val_labels


def class_of(path, kind, root):
    name = os.path.basename(path)
    if kind == "ai":
        m = AI_RE.match(name)
        return int(m.group(1)) if m else None
    m = WNID_RE.search(os.path.relpath(path, root))
    if m:
        return WNID_TO_IDX.get(m.group(1))
    m = VAL_RE.search(name)
    if m:
        return WNID_TO_IDX.get(val_labels()[int(m.group(1)) - 1])
    return None


index = {}
for k in KEYS:
    by_class, unknown = {}, 0
    for p in files[k]:
        c = class_of(p, k[1], SPLIT_DIRS[k])
        if c is None:
            unknown += 1
        else:
            by_class.setdefault(c, []).append(p)
    index[k] = by_class
    n = np.array([len(v) for v in by_class.values()])
    print(f"{k}: {len(by_class)} classes, per class min/median/max = "
          f"{n.min()}/{int(np.median(n))}/{n.max()}, unlabelled files: {unknown}")

CLASSES = sorted(set.intersection(*(set(index[k]) for k in KEYS)))
if SMOKE_CLASSES:
    CLASSES = CLASSES[:SMOKE_CLASSES]
print(f"Using {len(CLASSES)} classes")

## 4. Helpers: pHash, real/synthetic checks, crop offsets
Real crops sit on the JPEG MCU grid (needed for a lossless crop). Synthetic crops sit off both the 8-pixel grid and the 4-pixel-shifted grid (offset mod 8 ∉ {0, 4}).

In [ ]:
def phash_u64(img):
    """64-bit DCT perceptual hash (same algorithm as imagehash.phash)."""
    a = np.asarray(img.convert("L").resize((32, 32), Image.LANCZOS), dtype=np.float64)
    low = dct(dct(a, axis=0), axis=1)[:8, :8]
    return int.from_bytes(np.packbits(low > np.median(low)).tobytes(), "big")


_POP8 = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
def popcount64(x):
    x = np.ascontiguousarray(x, dtype=np.uint64)
    if hasattr(np, "bitwise_count"):
        return np.bitwise_count(x)
    return _POP8[x.view(np.uint8)].reshape(x.shape + (8,)).sum(-1)


class HashPool:
    def __init__(self, cap=4096):
        self.h, self.n = np.zeros(cap, np.uint64), 0

    def min_dist(self, h):
        return 64 if self.n == 0 else int(popcount64(self.h[:self.n] ^ np.uint64(h)).min())

    def add(self, h):
        if self.n == len(self.h):
            self.h = np.concatenate([self.h, np.zeros_like(self.h)])
        self.h[self.n] = h
        self.n += 1


def inspect_real(path):
    """Accept a real JPEG only if it can be cropped losslessly and its tables re-used exactly."""
    try:
        with Image.open(path) as im:
            if im.format != "JPEG":
                return {"reason": f"format_{im.format}"}
            if im.mode != "RGB":
                return {"reason": f"mode_{im.mode}"}
            if im.info.get("progressive") or im.info.get("progression"):
                return {"reason": "progressive"}
            if im.info.get("adobe") is not None and im.info.get("adobe_transform") == 0:
                return {"reason": "rgb_colour_space"}
            w, h = im.size
            if min(w, h) < CROP:
                return {"reason": "short_side_lt_crop"}
            sampling = get_sampling(im)
            if sampling not in (0, 1, 2):
                return {"reason": "subsampling"}
            if sorted(im.quantization) != [0, 1] or [l[3] for l in im.layer] != [0, 1, 1]:
                return {"reason": "qtable_layout"}
            # Lossless crop needs the offset on the MCU grid (8 x max sampling factor).
            mw, mh = 8 * max(l[1] for l in im.layer), 8 * max(l[2] for l in im.layer)
            x = ((w - CROP) // 2) // mw * mw
            y = ((h - CROP) // 2) // mh * mh
            ph = phash_u64(im.crop((x, y, x + CROP, y + CROP)))
    except Exception as e:
        return {"reason": f"error_{type(e).__name__}"}
    return {"reason": None, "src_w": w, "src_h": h, "crop_x": x, "crop_y": y,
            "src_subsampling": sampling, "phash": ph}


def fake_offset(path, size):
    """Crop offset near the centre whose remainder mod 8 is in FAKE_OFFSET_RESIDUES on both axes.

    The residue is drawn per file (seeded by its name), so no single phase between the autoencoder's
    8-pixel pattern and the JPEG grid is shared by every synthetic image.
    """
    rng = np.random.default_rng([SEED, 11, zlib.crc32(os.path.basename(path).encode())])
    off = []
    for n in size:
        base = ((n - CROP) // 2) // 8 * 8
        r = int(rng.choice(FAKE_OFFSET_RESIDUES))
        o = base + r if base + r <= n - CROP else base - 8 + r
        if o < 0:
            return None
        off.append(o)
    return tuple(off)


def inspect_fake(path):
    try:
        with Image.open(path) as im:
            if im.format != "PNG":
                return {"reason": f"format_{im.format}"}
            w, h = im.size
            if min(w, h) < CROP:
                return {"reason": "short_side_lt_crop"}
            off = fake_offset(path, (w, h))
            if off is None:
                return {"reason": "no_valid_offset"}
            x, y = off
            ph = phash_u64(im.convert("RGB").crop((x, y, x + CROP, y + CROP)))
    except Exception as e:
        return {"reason": f"error_{type(e).__name__}"}
    return {"reason": None, "src_w": w, "src_h": h, "crop_x": x, "crop_y": y, "phash": ph}

## 5. Pass 1: select the shared identifier list
For each split, label and class, candidates are visited in a seeded random order until the quota is filled. A candidate is skipped if it fails the checks above or is a near-duplicate (pHash distance ≤ `PHASH_MAX_DIST`) of **any** image already selected, across classes, labels and splits, so train/val leakage is also excluded.

GenImage's val folders hold only a few images per class, so after the checks some val classes cannot reach the quota. With `FILL_VAL_FROM_TRAIN`, such a class is topped up from the same class and label in the GenImage train folders, visited after all val candidates in a separately seeded order. Classes with enough val candidates select exactly the images they would without the fallback. Every source file is used at most once, and the manifest column `src_split` records which folder each image came from.

In [ ]:
SPLIT_ID = {"train": 0, "val": 1}
KIND_LABEL = {"nature": 0, "ai": 1}

pool, rows, rejects, short, used, filled = HashPool(), [], Counter(), [], set(), Counter()
for split in ("train", "val"):
    need = N_PER_CLASS[split]
    for kind in ("nature", "ai"):
        label = KIND_LABEL[kind]
        inspect = inspect_real if label == 0 else inspect_fake
        for c in tqdm(CLASSES, desc=f"select {split}/{kind}"):
            cands = index[(split, kind)].get(c, [])
            rng = np.random.default_rng([SEED, SPLIT_ID[split], label, c])
            order = [(split, cands[i]) for i in rng.permutation(len(cands))]
            if split == "val" and FILL_VAL_FROM_TRAIN:
                extra = index[("train", kind)].get(c, [])
                rng_fill = np.random.default_rng([SEED, 2, label, c])
                order += [("train", extra[i]) for i in rng_fill.permutation(len(extra))]
            got = 0
            for src_split, path in order:
                if got == need:
                    break
                if path in used:  # already selected for train
                    continue
                tag = kind if src_split == split else f"{kind}_fill_from_train"
                info = inspect(path)
                reason = info.pop("reason")
                if reason:
                    rejects[(split, tag, reason)] += 1
                    continue
                if pool.min_dist(info["phash"]) <= PHASH_MAX_DIST:
                    rejects[(split, tag, "near_duplicate")] += 1
                    continue
                pool.add(info["phash"])
                used.add(path)
                rows.append({"split": split, "label": label, "class_idx": c,
                             "src_split": src_split, "src_path": path, **info})
                got += 1
                if src_split != split:
                    filled[(split, kind)] += 1
            if got < need:
                short.append((split, kind, c, got))

print(f"Selection took {time.time() - T0:.0f}s")
print("val images filled from the train folders:", dict(filled) or "none")

The rejection log is written before checking for shortfalls so that it is available even when the cell raises.

In [ ]:
rej = pd.DataFrame([(*k, v) for k, v in rejects.items()], columns=["split", "kind", "reason", "count"])
rej.to_csv(OUT / "selection_rejects.csv", index=False)
print(rej.sort_values(["split", "kind", "count"], ascending=[True, True, False]).to_string(index=False))
if short:
    raise RuntimeError(f"{len(short)} (split, kind, class) groups lack enough usable images, "
                       f"e.g. {short[:10]}")

sel = pd.DataFrame(rows)
sel["id"] = sel["split"] + "_" + sel["src_path"].map(lambda p: Path(p).stem)
assert sel["id"].is_unique, "duplicate image ids"
sel["phash"] = sel["phash"].map(lambda h: f"{h:016x}")
sel.groupby(["split", "label"]).size()

## 6. Pair every synthetic image with a distinct real image of the same class
One random permutation per (split, class) gives a one-to-one pairing, so the set of quantization tables used by the D-align synthetic images is exactly the set used by the reals.

In [ ]:
sel["paired_real_id"] = None
for (split, c), g in sel.groupby(["split", "class_idx"], sort=True):
    reals = g.loc[g["label"] == 0, "id"].tolist()
    fakes = g.index[g["label"] == 1]
    assert len(reals) == len(fakes) == N_PER_CLASS[split], (split, c)
    perm = np.random.default_rng([SEED, 7, SPLIT_ID[split], c]).permutation(len(reals))
    sel.loc[fakes, "paired_real_id"] = [reals[j] for j in perm]


def out_rel(r, folder, ext):
    return f"{folder}/{r['split']}/{r['class_idx']:03d}/{r['id']}.{ext}"

is_real = sel["label"] == 0
sel["path_dbias"] = [out_rel(r, "real", "jpg") if r["label"] == 0 else out_rel(r, "fake_png", "png")
                     for _, r in sel.iterrows()]
sel["path_dalign"] = [out_rel(r, "real", "jpg") if r["label"] == 0 else out_rel(r, "fake_jpeg", "jpg")
                      for _, r in sel.iterrows()]
REAL_PATH = dict(zip(sel.loc[is_real, "id"], sel.loc[is_real, "path_dbias"]))
assert sel.loc[~is_real, "paired_real_id"].is_unique

## 7. Pass 2: write the images
- **Real:** `jpegtran -copy none -crop` on the MCU grid, so DCT coefficients, quantization tables and subsampling are untouched and all metadata is dropped. Each output is checked: 336×336, same tables and subsampling as the source, same decoded pixels (away from the border, where chroma upsampling sees different neighbours), no EXIF/ICC/XMP/IPTC/comment segments.
- **Synthetic:** crop at the off-grid offset from step 4 into a fresh image (no PNG text chunks, ICC or EXIF carried over), then saved as PNG (D-bias) and as JPEG with the paired real's tables and subsampling (D-align). The JPEG's APP segments (JFIF density, Adobe colour-transform flag) are replaced byte-for-byte with those of the paired real, so the two classes do not differ in header fields either; decoded pixels are checked to be unchanged by this. Both files are read back and checked.

In [ ]:
def jpeg_markers(path):
    """Marker codes of the segments before the first SOS."""
    data = Path(path).read_bytes()
    i, out = 2, []
    while i + 4 <= len(data) and data[i] == 0xFF:
        m = data[i + 1]
        if m == 0xFF:
            i += 1
            continue
        out.append(m)
        if m == 0xDA:
            break
        i += 2 + int.from_bytes(data[i + 2:i + 4], "big")
    return out


METADATA_MARKERS = {0xE1, 0xE2, 0xED, 0xFE}  # EXIF/XMP, ICC, IPTC, comment


def app_segments(data):
    """(bytes of the APPn segments directly after SOI, offset where the first non-APPn segment starts)."""
    i = 2
    while i + 4 <= len(data) and data[i] == 0xFF and 0xE0 <= data[i + 1] <= 0xEF:
        i += 2 + int.from_bytes(data[i + 2:i + 4], "big")
    return data[2:i], i


def copy_app_segments(src, dst):
    """Replace dst's APPn segments with src's. jpegtran keeps the source's JFIF density (often 72 dpi)
    and Adobe marker, while Pillow writes density 1:1 and no Adobe marker; this removes that header cue."""
    s, d = Path(src).read_bytes(), Path(dst).read_bytes()
    s_app, _ = app_segments(s)
    _, d_end = app_segments(d)
    Path(dst).write_bytes(d[:2] + s_app + d[d_end:])


def make_real(r):
    dst = OUT / r["path_dbias"]
    dst.parent.mkdir(parents=True, exist_ok=True)
    x, y = r["crop_x"], r["crop_y"]
    res = subprocess.run([JPEGTRAN, "-copy", "none", "-perfect", "-crop", f"{CROP}x{CROP}+{x}+{y}",
                          "-outfile", str(dst), r["src_path"]],
                         capture_output=True, stdin=subprocess.DEVNULL)
    if res.returncode:
        return f"jpegtran: {res.stderr.decode(errors='replace').strip()}"
    with Image.open(r["src_path"]) as a, Image.open(dst) as b:
        if b.size != (CROP, CROP):
            return f"size {b.size}"
        if b.quantization != a.quantization:
            return "quantization tables changed"
        if get_sampling(b) != get_sampling(a):
            return "subsampling changed"
        if CHECK_PIXELS:
            pa = np.asarray(a.convert("RGB").crop((x, y, x + CROP, y + CROP)))[16:-16, 16:-16]
            pb = np.asarray(b.convert("RGB"))[16:-16, 16:-16]
            if not np.array_equal(pa, pb):
                return "decoded pixels differ: crop was not lossless"
    left = METADATA_MARKERS & set(jpeg_markers(dst))
    if left:
        return f"metadata segments left: {sorted(hex(m) for m in left)}"
    return None


def make_fake(r):
    png, jpg = OUT / r["path_dbias"], OUT / r["path_dalign"]
    png.parent.mkdir(parents=True, exist_ok=True)
    jpg.parent.mkdir(parents=True, exist_ok=True)
    x, y = r["crop_x"], r["crop_y"]
    with Image.open(r["src_path"]) as im:
        pixels = np.asarray(im.convert("RGB").crop((x, y, x + CROP, y + CROP)))
    img = Image.fromarray(pixels)
    img.save(png, format="PNG")
    with Image.open(OUT / REAL_PATH[r["paired_real_id"]]) as ref:
        qtables, sampling = ref.quantization, get_sampling(ref)
    # No `quality` argument: with explicit qtables Pillow then uses them unscaled.
    img.save(jpg, format="JPEG", qtables=qtables, subsampling=sampling,
             optimize=False, progressive=False)
    with Image.open(jpg) as j:
        before = np.asarray(j.convert("RGB"))
    copy_app_segments(OUT / REAL_PATH[r["paired_real_id"]], jpg)
    with Image.open(png) as p:
        if not np.array_equal(np.asarray(p.convert("RGB")), pixels):
            return "PNG round trip changed pixels"
    with Image.open(jpg) as j:
        if j.size != (CROP, CROP):
            return f"size {j.size}"
        if j.quantization != qtables or [l[3] for l in j.layer] != [0, 1, 1]:
            return "quantization tables not reproduced"
        if get_sampling(j) != sampling:
            return "subsampling not reproduced"
        if j.info.get("progressive"):
            return "progressive"
        if not np.array_equal(np.asarray(j.convert("RGB")), before):
            return "copying APP segments changed decoded pixels"
    real_app = app_segments((OUT / REAL_PATH[r["paired_real_id"]]).read_bytes())[0]
    if app_segments(jpg.read_bytes())[0] != real_app:
        return "APP segments differ from the paired real"
    return None


def run_all(fn, df, desc):
    def safe(r):
        try:
            return fn(r)
        except Exception as e:
            return f"{type(e).__name__}: {e}"
    recs = df.to_dict("records")
    with ThreadPoolExecutor(N_WORKERS) as ex:
        results = list(tqdm(ex.map(safe, recs), total=len(recs), desc=desc))
    errors = {r["id"]: e for r, e in zip(recs, results) if e}
    if errors:
        for k, e in list(errors.items())[:20]:
            print(k, "->", e)
        raise RuntimeError(f"{desc}: {len(errors)} of {len(recs)} images failed verification")


for sub in ("real", "fake_png", "fake_jpeg"):
    shutil.rmtree(OUT / sub, ignore_errors=True)

run_all(make_real, sel[is_real], "real (lossless crop)")
run_all(make_fake, sel[~is_real], "synthetic (PNG + JPEG)")
print(f"Images written after {time.time() - T0:.0f}s")

## 8. Manifest, loader CSVs and integrity checks

In [ ]:
def sha256(rel):
    return hashlib.sha256((OUT / rel).read_bytes()).hexdigest()

with ThreadPoolExecutor(N_WORKERS) as ex:
    sel["sha256_dbias"] = list(ex.map(sha256, sel["path_dbias"]))
    sel["sha256_dalign"] = list(ex.map(sha256, sel["path_dalign"]))

cols = ["id", "split", "label", "class_idx", "src_split", "src_path", "src_w", "src_h", "crop_x", "crop_y",
        "src_subsampling", "phash", "paired_real_id", "path_dbias", "path_dalign",
        "sha256_dbias", "sha256_dalign"]
sel = sel.sort_values(["split", "label", "class_idx", "id"]).reset_index(drop=True)
sel["src_subsampling"] = sel["src_subsampling"].astype("Int64")
sel[cols].to_csv(OUT / "manifest.csv", index=False)
for version, col in (("d_bias", "path_dbias"), ("d_align", "path_dalign")):
    sel[["id", col, "label", "split", "class_idx"]].rename(columns={col: "path"}) \
        .to_csv(OUT / f"{version}.csv", index=False)

# Exact counts per split, label and class
counts = sel.groupby(["split", "label", "class_idx"]).size()
for split, n in N_PER_CLASS.items():
    assert (counts.loc[split] == n).all() and len(counts.loc[split]) == 2 * len(CLASSES), split
print(sel.groupby(["split", "label"]).size().rename("images").to_string())

# Synthetic crop offsets are off both the 8-pixel grid and the 4-pixel-shifted grid
fk = sel[sel.label == 1]
assert (~(fk[["crop_x", "crop_y"]] % 8).isin([0, 4])).all().all()
print("synthetic crop offset mod 8:", Counter(zip(fk.crop_x % 8, fk.crop_y % 8)).most_common(5), "...")

# Real files are byte-identical in both versions; every real is paired exactly once
assert (sel.loc[sel.label == 0, "sha256_dbias"] == sel.loc[sel.label == 0, "sha256_dalign"]).all()
assert set(sel.loc[sel.label == 1, "paired_real_id"]) == set(sel.loc[sel.label == 0, "id"])

# Smallest pHash distance between any two selected images
H = np.array([int(h, 16) for h in sel["phash"]], dtype=np.uint64)
min_d = 64
for i in range(0, len(H), 256):
    d = popcount64(H[i:i + 256, None] ^ H[None, :]).astype(np.int16)
    d[np.arange(d.shape[0]), np.arange(i, i + d.shape[0])] = 64
    min_d = min(min_d, int(d.min()))
print(f"min pairwise pHash distance: {min_d} (threshold {PHASH_MAX_DIST})")
assert min_d > PHASH_MAX_DIST


# Header details that the metadata audit does not use, compared between real and D-align synthetic files
def header_info(rel):
    with Image.open(OUT / rel) as im:
        return {"jfif_density": str(im.info.get("jfif_density")), "jfif_unit": im.info.get("jfif_unit"),
                "kb": (OUT / rel).stat().st_size / 1024}

hdr = pd.DataFrame([header_info(p) for p in sel["path_dalign"]])
hdr["group"] = np.where(sel["label"] == 0, "real", "fake_jpeg")
print(hdr.groupby("group")["kb"].describe().round(1).to_string())
print(hdr.groupby("group")[["jfif_density", "jfif_unit"]].value_counts().head(10).to_string())

## 9. Encoding check 1: metadata
A logistic regression on metadata alone (format, estimated quality factor, quantization-table hash, subsampling, width, height) is trained on the train split and scored on the held-out val split, separately for each version. Pass criteria: accuracy **≥ 0.95 on D-bias** (the bias is present) and **≤ 0.55 on D-align** (the listed encoding properties no longer separate the classes). If D-align fails, the experiment still runs, but its results are read as a *reduction* rather than a *removal* of encoding bias.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Reference luminance tables for IJG quality 1..100, generated with the same encoder,
# so the estimate does not depend on the table ordering Pillow uses.
_REF_Q = []
for q in range(1, 101):
    buf = io.BytesIO()
    Image.new("RGB", (16, 16)).save(buf, format="JPEG", quality=q)
    with Image.open(buf) as im:
        _REF_Q.append(list(im.quantization[0]))
_REF_Q = np.array(_REF_Q, dtype=np.int32)


def estimate_qf(lum_table):
    return int(np.abs(_REF_Q - np.asarray(lum_table, dtype=np.int32)).sum(1).argmin()) + 1


def meta_features(rel):
    with Image.open(OUT / rel) as im:
        f = {"path": rel, "format": im.format, "width": im.size[0], "height": im.size[1]}
        if im.format == "JPEG":
            q = {k: list(v) for k, v in sorted(im.quantization.items())}
            f.update(qf=estimate_qf(q[0]), subsampling=str(get_sampling(im)),
                     qtable_hash=hashlib.sha1(json.dumps(q).encode()).hexdigest()[:12])
        else:
            f.update(qf=-1, subsampling="none", qtable_hash="none")
    return f


paths = sorted(set(sel["path_dbias"]) | set(sel["path_dalign"]))
with ThreadPoolExecutor(N_WORKERS) as ex:
    feats = pd.DataFrame(list(tqdm(ex.map(meta_features, paths), total=len(paths), desc="metadata")))
feats.to_csv(OUT / "metadata_features.csv", index=False)

CAT, NUM = ["format", "qtable_hash", "subsampling"], ["qf", "width", "height"]
CRITERIA = {"d_bias": (">=", 0.95), "d_align": ("<=", 0.55)}
audit = {}
for version, col in (("d_bias", "path_dbias"), ("d_align", "path_dalign")):
    df = sel[["split", "label", col]].merge(feats, left_on=col, right_on="path")
    tr, va = df[df.split == "train"], df[df.split == "val"]
    model = make_pipeline(
        ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
                           ("num", StandardScaler(), NUM)]),
        LogisticRegression(max_iter=2000))
    model.fit(tr[CAT + NUM], tr["label"])
    acc_tr = float((model.predict(tr[CAT + NUM]) == tr["label"]).mean())
    acc_va = float((model.predict(va[CAT + NUM]) == va["label"]).mean())
    op, thr = CRITERIA[version]
    a = round(acc_va, 6)  # avoid float error at the boundary (e.g. 0.55 stored as 0.5500000000000002)
    passed = a >= thr if op == ">=" else a <= thr
    audit[version] = {"train_acc": acc_tr, "val_acc": acc_va, "criterion": f"{op} {thr}", "pass": passed}
    print(f"{version:8s} val acc {acc_va:.3f} (train {acc_tr:.3f}), needs {op} {thr}: "
          f"{'PASS' if passed else 'FAIL'}")

if not audit["d_align"]["pass"]:
    print("WARNING: D-align fails the metadata check; read its results as a reduction, "
          "not a removal, of encoding bias.")
if not audit["d_bias"]["pass"]:
    print("WARNING: D-bias does not reproduce the documented format bias.")
(OUT / "audit_metadata.json").write_text(json.dumps(audit, indent=2))

## 10. Encoding check 2: JPEG features on the block grid vs a shifted grid
A classifier on pixels would also pick up generation artifacts, so its accuracy alone cannot be read as encoding bias. JPEG traces are tied to the 8×8 block grid and generation artifacts are not, so the same JPEG features are computed twice on the luminance channel: once on the 8×8 grid of the file and once on that grid shifted by `GRID_SHIFT` = 4 px on both axes. Both use the same 41×41 blocks of area.

Features per grid:
- **Blockiness:** mean |pixel difference| across block boundaries ÷ mean |difference| inside blocks, horizontal and vertical (2 features).
- **DCT coefficient statistics:** orthonormal 8×8 DCT of every block (the transform JPEG uses). For each of the 63 AC frequencies: the fraction of coefficients with |c| < 0.5 and the mean log(1 + |c|) (126 features).

One logistic regression is trained per grid on train and scored on val. accuracy(grid) − accuracy(shifted) estimates the contribution of encoding. **D-align passes if it is ≤ 0.05**; the D-bias value is reported for comparison.

**Positive control (reported, not a pass criterion):** PNG and JPEG differ on *both* grids, so on D-bias the two accuracies can both sit near 1.0 and their difference near 0, which would not show that the check can detect a known bias. A third classifier on the per-image difference (grid − shifted features) sees only grid-locked traces; it should be high on D-bias and near 0.5 on D-align.

**Limitation of the difference criterion.** A lossy pass also removes high-frequency energy everywhere, not only on the block grid, so an earlier compression of the real images (e.g. by the photo-sharing site, before ImageNet) is visible on *both* grids and cancels out of the difference. In a simulation where reals carried one extra compression, D-align passed the ≤ 0.05 criterion (difference +0.011) while the grid-minus-shifted classifier separated the classes at 0.94. The cell therefore also warns when the D-align grid-minus-shifted accuracy exceeds `GRID_CONTROL_MAX` or when both D-align grid accuracies are at ceiling; neither warning changes the pre-specified pass/fail.

In [ ]:
from scipy.fft import dctn

GRID_BLOCKS = (CROP - GRID_SHIFT) // 8  # 41 blocks per side on both grids
GRID_CONTROL_MAX = 0.55  # warn if grid-locked features alone separate the D-align classes above this


def grid_features(y, off):
    """Blockiness + per-frequency DCT statistics of luminance y on the 8x8 grid starting at (off, off)."""
    dh = np.abs(np.diff(y, axis=1))  # dh[:, j]: between columns j and j+1
    dv = np.abs(np.diff(y, axis=0))
    edge = (np.arange(y.shape[1] - 1) + 1 - off) % 8 == 0  # j+1 is a grid line
    f = [dh[:, edge].mean() / (dh[:, ~edge].mean() + 1e-6),
         dv[edge].mean() / (dv[~edge].mean() + 1e-6)]
    n = GRID_BLOCKS * 8
    blocks = y[off:off + n, off:off + n].reshape(GRID_BLOCKS, 8, GRID_BLOCKS, 8).transpose(0, 2, 1, 3)
    c = np.abs(dctn(blocks - 128.0, axes=(2, 3), norm="ortho").reshape(-1, 64)[:, 1:])
    return np.concatenate([f, (c < 0.5).mean(0), np.log1p(c).mean(0)])


def both_grids(rel):
    with Image.open(OUT / rel) as im:
        y = np.asarray(im.convert("YCbCr"), dtype=np.float64)[..., 0]
    return grid_features(y, 0), grid_features(y, GRID_SHIFT)


with ThreadPoolExecutor(N_WORKERS) as ex:
    gf = list(tqdm(ex.map(both_grids, paths), total=len(paths), desc="grid features"))
GF = {"grid": np.stack([g[0] for g in gf]), "shifted": np.stack([g[1] for g in gf])}
np.savez_compressed(OUT / "grid_features.npz", paths=np.array(paths), **GF)
ROW = {p: i for i, p in enumerate(paths)}

grid_audit = {}
for version, col in (("d_bias", "path_dbias"), ("d_align", "path_dalign")):
    idx = sel[col].map(ROW).to_numpy()
    tr, va = (sel["split"] == "train").to_numpy(), (sel["split"] == "val").to_numpy()
    y_lab = sel["label"].to_numpy()
    acc = {}
    # grid_minus_shifted sees only grid-locked traces: a positive control that does not saturate on D-bias
    for grid, X in {**GF, "grid_minus_shifted": GF["grid"] - GF["shifted"]}.items():
        model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
        model.fit(X[idx[tr]], y_lab[tr])
        acc[grid] = float((model.predict(X[idx[va]]) == y_lab[va]).mean())
    diff = round(acc["grid"] - acc["shifted"], 6)  # rounded: 0.55 - 0.50 is 0.05000000000000004 in floating point
    grid_audit[version] = {"val_acc_grid": acc["grid"], "val_acc_shifted": acc["shifted"], "difference": diff,
                           "val_acc_grid_minus_shifted": acc["grid_minus_shifted"]}
    line = (f"{version:8s} val acc grid {acc['grid']:.3f}, shifted {acc['shifted']:.3f}, difference {diff:+.3f}"
            f" | grid-minus-shifted features {acc['grid_minus_shifted']:.3f}")
    if version == "d_align":
        grid_audit[version].update(criterion="<= 0.05", **{"pass": diff <= 0.05})
        line += f", needs <= 0.05: {'PASS' if diff <= 0.05 else 'FAIL'}"
    print(line)

if min(grid_audit["d_bias"]["val_acc_grid"], grid_audit["d_bias"]["val_acc_shifted"]) >= 0.99:
    print("NOTE: on D-bias both grids are at ceiling, so their difference cannot show the known bias. "
          "Use the grid-minus-shifted accuracy on D-bias as the positive control (it should be high).")
da = grid_audit["d_align"]
da["control_flag"] = round(da["val_acc_grid_minus_shifted"], 6) > GRID_CONTROL_MAX
da["ceiling_flag"] = min(da["val_acc_grid"], da["val_acc_shifted"]) >= 0.95
if da["control_flag"]:
    print(f"WARNING: grid-locked features still separate the D-align classes "
          f"({da['val_acc_grid_minus_shifted']:.3f} > {GRID_CONTROL_MAX}); the difference criterion cannot see "
          "compression applied to the reals before this build. Report this alongside the pass/fail.")
if da["ceiling_flag"]:
    print("WARNING: on D-align both grid classifiers are at >= 0.95, so their difference is uninformative.")
if not grid_audit["d_align"]["pass"]:
    print("WARNING: D-align fails the grid-shift check; read its results as a reduction, "
          "not a removal, of encoding bias.")
(OUT / "audit_grid_shift.json").write_text(json.dumps(grid_audit, indent=2))

## 11. Look at a few examples

In [ ]:
import matplotlib.pyplot as plt

ex = sel[(sel.split == "train") & (sel.label == 1)].head(4)
fig, axes = plt.subplots(len(ex), 3, figsize=(9, 3 * len(ex)))
for ax_row, (_, r) in zip(np.atleast_2d(axes), ex.iterrows()):
    for ax, rel_path, title in zip(ax_row,
                                   [REAL_PATH[r["paired_real_id"]], r["path_dbias"], r["path_dalign"]],
                                   ["paired real", "D-bias (PNG)", "D-align (JPEG)"]):
        ax.imshow(Image.open(OUT / rel_path))
        ax.set_title(f"{title}\nclass {r['class_idx']}", fontsize=9)
        ax.axis("off")
plt.tight_layout()
plt.show()

## 12. Summary
After the full build (`SMOKE_CLASSES = None`): **Save Version → Save & Run All**, then use the notebook output as a new Kaggle dataset and attach it to the training notebooks. Loaders should read `d_bias.csv` or `d_align.csv`; the `path` column is relative to the dataset root.

In [ ]:
size_gb = sum(f.stat().st_size for f in OUT.rglob("*") if f.is_file()) / 1e9
BUILD_INFO.update(
    n_classes=len(CLASSES),
    counts={f"{s}_{'fake' if l else 'real'}": int(n) for (s, l), n in sel.groupby(["split", "label"]).size().items()},
    val_filled_from_train={f"{s}_{k}": int(n) for (s, k), n in filled.items()},
    min_phash_distance=min_d, audit=audit, audit_grid_shift=grid_audit, output_gb=round(size_gb, 2),
    minutes=round((time.time() - T0) / 60, 1))
(OUT / "build_info.json").write_text(json.dumps(BUILD_INFO, indent=2))
print(json.dumps(BUILD_INFO, indent=2))